In [1]:
import numpy as np


def min_edit_distance_nlp(source: str, target: str) -> np.ndarray:
    """Calculates the Minimum Edit Distance matrix using standard NLP costs

    (del=1, ins=1, sub=2).
    """
    n, m = len(source), len(target)

    # Initialize matrix D with zeros
    D = np.zeros((n + 1, m + 1), dtype=int)

    # Base cases: populate boundary rows/columns
    for i in range(1, n + 1):
        D[i][0] = D[i - 1][0] + 1  # Deletions

    for j in range(1, m + 1):
        D[0][j] = D[0][j - 1] + 1  # Insertions

    # Fill matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if source[i - 1] == target[j - 1]:
                D[i][j] = D[i - 1][j - 1]  # No operation needed
            else:
                D[i][j] = min(
                    D[i - 1][j] + 1,  # Deletion
                    D[0][j - 1] + 1 if i == 0 else D[i][j - 1] + 1,  # Insertion
                    D[i - 1][j - 1] + 2,  # Substitution (Cost = 2)
                )

    return D

In [2]:
# Classic NLP benchmark pair: "intention" -> "execution"
src, tgt = "intention", "execution"
matrix = min_edit_distance_nlp(src, tgt)

print(f"Source: {src}")
print(f"Target: {tgt}")
print(f"Minimum Edit Distance: {matrix[-1, -1]}")

Source: intention
Target: execution
Minimum Edit Distance: 8


In [3]:
# QWERTY Key Coordinates for Spatial Distance
QWERTY_GRID = {
    'q': (0, 0),
    'w': (0, 1),
    'e': (0, 2),
    'r': (0, 3),
    't': (0, 4),
    'y': (0, 5),
    'u': (0, 6),
    'i': (0, 7),
    'o': (0, 8),
    'p': (0, 9),
    'a': (1, 0),
    's': (1, 1),
    'd': (1, 2),
    'f': (1, 3),
    'g': (1, 4),
    'h': (1, 5),
    'j': (1, 6),
    'k': (1, 7),
    'l': (1, 8),
    'z': (2, 0),
    'x': (2, 1),
    'c': (2, 2),
    'v': (2, 3),
    'b': (2, 4),
    'n': (2, 5),
    'm': (2, 6),
}


def keyboard_sub_cost(c1: str, c2: str) -> float:
    """Calculates substitution cost based on physical key distance."""
    if c1 == c2:
        return 0.0

    c1, c2 = c1.lower(), c2.lower()
    if c1 in QWERTY_GRID and c2 in QWERTY_GRID:
        x1, y1 = QWERTY_GRID[c1]
        x2, y2 = QWERTY_GRID[c2]
        # Euclidean distance normalized between 0.5 and 2.0
        dist = np.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)
        return min(2.0, max(0.5, float(dist)))

    return 2.0  # Default non-keyboard fallback


def weighted_edit_distance_nlp(
    source: str,
    target: str,
    del_cost: float = 1.0,
    ins_cost: float = 1.0,
    sub_cost_fn=keyboard_sub_cost,
) -> float:
    """Computes dynamic-programming weighted edit distance for NLP tasks."""
    n, m = len(source), len(target)
    D = np.zeros((n + 1, m + 1), dtype=float)

    for i in range(1, n + 1):
        D[i][0] = D[i - 1][0] + del_cost
    for j in range(1, m + 1):
        D[0][j] = D[0][j - 1] + ins_cost

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s_char = source[i - 1]
            t_char = target[j - 1]

            cost_del = D[i - 1][j] + del_cost
            cost_ins = D[i][j - 1] + ins_cost
            cost_sub = D[i - 1][j - 1] + sub_cost_fn(s_char, t_char)

            D[i][j] = min(cost_del, cost_ins, cost_sub)

    return D[n][m]

In [4]:
# Vocabulary dictionary
VOCABULARY = ["receive", "receipt", "recite", "deceive", "reaction"]

misspelled_word = "recieve"  # Typo: 'i' and 'e' swapped

print(f"Query Word: '{misspelled_word}'\n")
print("Ranking Candidate Corrections:")
print("-" * 45)

results = []
for candidate in VOCABULARY:
    dist = weighted_edit_distance_nlp(misspelled_word, candidate)
    results.append((candidate, dist))

# Sort by lowest edit distance
results.sort(key=lambda x: x[1])

for rank, (cand, cost) in enumerate(results, 1):
    print(f"{rank}. {cand:<12} | Weighted Distance Cost: {cost:.2f}")

Query Word: 'recieve'

Ranking Candidate Corrections:
---------------------------------------------
1. receive      | Weighted Distance Cost: 2.00
2. recite       | Weighted Distance Cost: 3.00
3. deceive      | Weighted Distance Cost: 3.41
4. receipt      | Weighted Distance Cost: 6.00
5. reaction     | Weighted Distance Cost: 7.00


In [5]:
def align_nlp_strings(source: str, target: str):
    """Backtracks through DP table to generate explicit NLP edit operation string."""
    D = min_edit_distance_nlp(source, target)
    i, j = len(source), len(target)

    align_src, align_tgt, ops = [], [], []

    while i > 0 or j > 0:
        if i > 0 and j > 0 and source[i - 1] == target[j - 1]:
            align_src.append(source[i - 1])
            align_tgt.append(target[j - 1])
            ops.append(" ")
            i -= 1
            j -= 1
        elif i > 0 and j > 0 and D[i][j] == D[i - 1][j - 1] + 2:
            align_src.append(source[i - 1])
            align_tgt.append(target[j - 1])
            ops.append("s")  # Substitution
            i -= 1
            j -= 1
        elif i > 0 and D[i][j] == D[i - 1][j] + 1:
            align_src.append(source[i - 1])
            align_tgt.append("*")
            ops.append("d")  # Deletion
            i -= 1
        else:
            align_src.append("*")
            align_tgt.append(target[j - 1])
            ops.append("i")  # Insertion
            j -= 1

    align_src.reverse()
    align_tgt.reverse()
    ops.reverse()

    print("Source:    " + " ".join(align_src))
    print("Operation: " + " ".join(ops))
    print("Target:    " + " ".join(align_tgt))


print("=== Alignment String Output ===")
align_nlp_strings("intention", "execution")

=== Alignment String Output ===
Source:    i n t e * n t i o n
Operation: d s s   i s        
Target:    * e x e c u t i o n


In [10]:
import pandas as pd


def display_edit_matrix(source: str, target: str):
    """Renders the DP matrix with character labels for visual debugging."""
    D = min_edit_distance_nlp(source, target)

    source_labels = ["#"] + list(source)
    target_labels = ["#"] + list(target)

    df = pd.DataFrame(D, index=source_labels, columns=target_labels)
    print(f"=== DP Cost Matrix for '{source}' -> '{target}' ===")
    return df


# Display matrix for short string pair
display_edit_matrix("drive", "brief")

=== DP Cost Matrix for 'drive' -> 'brief' ===


,#,b,r,i,e,f
#,0,1,2,3,4,5
d,1,2,3,4,5,6
r,2,3,2,3,4,5
i,3,4,3,2,3,4
v,4,5,4,3,4,5
e,5,6,5,4,3,4


In [7]:
def calculate_wer(reference: str, hypothesis: str) -> dict:
    """Calculates Word Error Rate (WER) between reference and hypothesis transcripts."""
    ref_words = reference.split()
    hyp_words = hypothesis.split()

    r_len, h_len = len(ref_words), len(hyp_words)
    D = np.zeros((r_len + 1, h_len + 1), dtype=int)

    for i in range(1, r_len + 1):
        D[i][0] = i
    for j in range(1, h_len + 1):
        D[0][j] = j

    for i in range(1, r_len + 1):
        for j in range(1, h_len + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                D[i][j] = D[i - 1][j - 1]
            else:
                D[i][j] = min(
                    D[i - 1][j] + 1,  # Deletion
                    D[i][j - 1] + 1,  # Insertion
                    D[i - 1][j - 1] + 1,  # Substitution
                )

    total_edits = D[r_len][h_len]
    wer_score = total_edits / max(1, r_len)

    return {
        "Reference Count": r_len,
        "Total Edit Distance": total_edits,
        "WER": round(wer_score, 4),
    }


# Test ASR Output Evaluation
ref_transcript = "the quick brown fox jumps over the lazy dog"
hyp_transcript = "the thick brown fox jumped over a lazy dog"

results = calculate_wer(ref_transcript, hyp_transcript)
print("=== ASR Evaluation Result ===")
for k, v in results.items():
    print(f"{k}: {v}")

=== ASR Evaluation Result ===
Reference Count: 9
Total Edit Distance: 3
WER: 0.3333
